# Jet flavour pie charts (parton level)

Presentation-friendly alternative to the 2D flavour content plots: for every sample and every jet flavour
definition, a pie chart of what the two exclusive ee_kt jets (hadronisation partons) get assigned. Built from
the `parton_<alg>_jets_category` histograms written by `jet-flv-test-histmaker.py` (one entry per jet).

* IFN and net summation (q = Higgs decay flavour, a quark counts +1, an antiquark -1):
  clean q (q +1, nothing else), clean q̄ (q -1, nothing else), no flavour, q + other flavour, other.
  For H → gg: no flavour (= clean), single flavour (one quark or antiquark, nothing else), other.
* `get_flavour` (`JetTaggingUtils::get_flavour`, no sign): q, gluon, unmatched, other quark.
* Slices below 2 % are not labelled on the pie; all percentages are in the legend.
* Layout for the right half of a 16:9 beamer slide (image at full column width): nearly square, legend below.
* Output: `plots/parton_<alg>/pie/<sample>.pdf`, one pie per PDF, plus `plots/parton_<alg>/pie/all_samples_pie.pdf`
  with all samples (one page each).

In [1]:
import os
import glob
import ROOT

ROOT.gROOT.SetBatch(True)
ROOT.gStyle.SetOptStat(0)

INPUT_DIR = os.path.join("..", "..", "..", "outputs", "jet-test")
PLOT_DIR = os.path.join(INPUT_DIR, "plots")
ALGOS = {"ifn": "IFN", "net": "net flavour", "getflav": "get_flavour"}
MIN_LABEL = 0.02  # slices below this fraction get no label on the pie

files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.root")))
files

['../../../outputs/jet-test/wzp6_ee_nunuH_Hbb_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hcc_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hdd_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hgg_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Hss_ecm240.root',
 '../../../outputs/jet-test/wzp6_ee_nunuH_Huu_ecm240.root']

In [2]:
def higgs_flavour(sample):
    """Higgs decay flavour from the sample name, e.g. 'b'; None for H -> gg."""
    decay = sample.split("_H")[-1].split("_")[0]
    return None if decay == "gg" else decay[0]


def categories(algo, q):
    """(bin index, label, colour) of the slices, in the order of the category codes of the histmaker."""
    if algo == "getflav":
        return [(0, q if q else "", ROOT.kAzure + 1),
                (1, "gluon", ROOT.kGray + 1),
                (2, "unmatched", ROOT.kGray),
                (3, "other quark" if q else "quark", ROOT.kRed - 4)]
    if q:
        return [(0, f"clean {q}", ROOT.kAzure + 1),
                (1, f"clean #bar{{{q}}}", ROOT.kAzure - 9),
                (2, "no flavour", ROOT.kGray + 1),
                (3, f"{q} + other flavour", ROOT.kOrange - 3),
                (4, "other", ROOT.kRed - 4)]
    return [(2, "no flavour", ROOT.kAzure + 1),
            (5, "single flavour", ROOT.kOrange - 3),
            (4, "other", ROOT.kRed - 4)]


def title(sample, algo):
    q = higgs_flavour(sample)
    decay = f"{q}#bar{{{q}}}" if q else "gg"
    return f"H #rightarrow {decay}, parton level, {ALGOS[algo]}"

In [3]:
def pct(f):
    """Percentage with more digits for tiny slices, so they do not show up as 0.0%."""
    return f"{100 * f:.1f}%" if f >= 0.001 else f"{100 * f:.2f}%"


def draw_pie(tfile, sample, algo):
    q = higgs_flavour(sample)
    h = tfile.Get(f"parton_{algo}_jets_category")
    n = h.Integral(0, h.GetNbinsX() + 1)
    slices = [(lab, col, h.GetBinContent(h.FindFixBin(i + 0.5)) / n) for i, lab, col in categories(algo, q)
              if (algo != "getflav" or q or i != 0)]
    assert abs(sum(f for _, _, f in slices) - 1.) < 1e-6, f"{sample} {algo}: categories do not add up"
    slices = [s for s in slices if s[2] > 0]

    # Layout for a half-width column of a 16:9 beamer slide (slightly wider than tall), in pixels from the bottom:
    # compact two-column legend, pie with room for the percentage labels above and below, title. TPie works in pad
    # fractions, so its pad must be square to get a round pie: it spans the full canvas height (centred, transparent,
    # overlapping title and legend), which also leaves room for the labels left and right of the pie. A pad must not
    # extend beyond the canvas, ROOT would shrink it and the pie becomes an ellipse.
    W = 600
    LEG_H = 120             # legend height
    LABEL_BELOW = 45        # space for the percentage labels below the pie
    LABEL_ABOVE = 55        # space above the pie: labels plus a gap to the title
    RADIUS = 148            # pie radius
    TEXT = 31               # title and pie label size; legend text slightly smaller
    y_pie = LEG_H + LABEL_BELOW + RADIUS
    y_title = y_pie + RADIUS + LABEL_ABOVE + TEXT / 2  # centre of the title
    H = int(y_title + 0.75 * TEXT)  # about 555: slightly wider than tall
    c = ROOT.TCanvas(f"pie_{algo}_{sample}", "", W, H)
    c.SetCanvasSize(W, H)  # exact drawing area (the constructor size includes the window decorations)
    p_pie = ROOT.TPad("p_pie", "", 0.5 - H / 2 / W, 0, 0.5 + H / 2 / W, 1)  # square: H x H pixels
    p_pie.SetFillStyle(0)
    p_leg = ROOT.TPad("p_leg", "", 0, 0, 1, LEG_H / H)
    for p in (p_pie, p_leg):
        p.SetMargin(0, 0, 0, 0)
        p.Draw()
    c.cd()
    tit = ROOT.TLatex(0.5, y_title / H, title(sample, algo))
    tit.SetNDC()
    tit.SetTextAlign(22)
    tit.SetTextFont(42)
    tit.SetTextSize(TEXT / min(W, H))  # text sizes are fractions of the smaller pad dimension
    tit.Draw()

    p_pie.cd()
    pie = ROOT.TPie(f"pie_{algo}_{sample}", "", len(slices))
    for i, (lab, col, f) in enumerate(slices):
        pie.SetEntryVal(i, f)
        pie.SetEntryFillColor(i, col)
        pie.SetEntryLabel(i, pct(f) if f >= MIN_LABEL else "")
    pie.SetLabelFormat("%txt")
    pie.SetCircle(0.5, y_pie / H, RADIUS / H)
    pie.SetLabelsOffset(0.015)
    pie.SetTextSize(TEXT / H)
    pie.Draw("nol")  # no 3D, labels horizontal

    p_leg.cd()
    leg = ROOT.TLegend(0.03, 0.02, 0.99, 0.98)
    leg.SetNColumns(2)
    leg.SetBorderSize(0)
    leg.SetFillStyle(0)
    leg.SetTextSize((TEXT - 3) / LEG_H)
    leg.SetMargin(0.12)
    for i, (lab, col, f) in enumerate(slices):
        leg.AddEntry(pie.GetSlice(i), f"{lab} ({pct(f)})", "f")
    leg.Draw()

    outdir = os.path.join(PLOT_DIR, f"parton_{algo}", "pie")
    os.makedirs(outdir, exist_ok=True)
    c.SaveAs(os.path.join(outdir, f"{sample}.pdf"))
    c.Draw()
    c._keep = (p_pie, p_leg, tit, pie, leg)
    return c, slices

In [4]:
canvases = {algo: [] for algo in ALGOS}
for fname in files:
    sample = os.path.basename(fname)[:-5]
    tfile = ROOT.TFile.Open(fname)
    for algo in ALGOS:
        c, slices = draw_pie(tfile, sample, algo)
        canvases[algo].append(c)
        print(f"{sample:28s} {algo:8s} " + ", ".join(f"{lab}: {pct(f)}" for lab, _, f in slices))
    tfile.Close()

# One multi-page overview PDF per flavour definition with all samples
for algo, cs in canvases.items():
    combined = os.path.join(PLOT_DIR, f"parton_{algo}", "pie", "all_samples_pie.pdf")
    cs[0].Print(combined + "[")
    for c in cs:
        c.Print(combined, f"Title:{c.GetName()}")
    cs[-1].Print(combined + "]")

Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created


Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_net/pie/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_getflav/pie/wzp6_ee_nunuH_Hbb_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_net/pie/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_getflav/pie/wzp6_ee_nunuH_Hcc_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_net/pie/wzp6_ee_nunuH_Hdd_ecm240.pdf has been created
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_getflav/pie/wzp6_ee_nu

Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf
Info in <TCanvas::Print>: Current canvas added to pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_ifn/pie/all_samples_pie.pdf has been closed
Info in <TCanvas::Print>: pdf file ../../../outputs/jet-test/plots/parton_net/pie/all_samples_pie.pdf has been created
Info in

wzp6_ee_nunuH_Hbb_ecm240     ifn      clean b: 47.8%, clean #bar{b}: 47.8%, no flavour: 4.4%, b + other flavour: 0.04%, other: 0.03%
wzp6_ee_nunuH_Hbb_ecm240     net      clean b: 48.0%, clean #bar{b}: 48.0%, no flavour: 3.4%, b + other flavour: 0.5%, other: 0.06%
wzp6_ee_nunuH_Hbb_ecm240     getflav  b: 95.8%, gluon: 1.6%, unmatched: 2.4%, other quark: 0.3%
wzp6_ee_nunuH_Hcc_ecm240     ifn      clean c: 47.5%, clean #bar{c}: 47.5%, no flavour: 5.0%, c + other flavour: 0.04%, other: 0.03%
wzp6_ee_nunuH_Hcc_ecm240     net      clean c: 47.9%, clean #bar{c}: 47.9%, no flavour: 3.8%, c + other flavour: 0.4%, other: 0.08%
wzp6_ee_nunuH_Hcc_ecm240     getflav  c: 95.7%, gluon: 1.8%, unmatched: 2.2%, other quark: 0.3%
wzp6_ee_nunuH_Hdd_ecm240     ifn      clean d: 47.3%, clean #bar{d}: 47.3%, no flavour: 5.3%, d + other flavour: 0.03%, other: 0.03%
wzp6_ee_nunuH_Hdd_ecm240     net      clean d: 47.7%, clean #bar{d}: 47.7%, no flavour: 4.1%, d + other flavour: 0.3%, other: 0.1%
wzp6_ee_nunuH_